# Solution 01: Zero-Shot Topic Classification

In [1]:
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer
import torch, json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Loaded {len(articles)} articles, device={device}")

✓ Loaded 10 articles, device=cuda


## Part 1: Load Model and Build Pipelines

In [2]:
model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)

pipeline_ml = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='multi-label', device=device
)
pipeline_sl = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='single-label', device=device
)

smoke = pipeline_ml("Apple released a new iPhone.", ["technology", "sports"], threshold=0.0)
assert isinstance(smoke, list) and isinstance(smoke[0], list)
assert all('label' in r and 'score' in r for r in smoke[0])
print("✓ Both pipelines loaded")

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.34s/it]

100%|██████████| 1/1 [00:02<00:00,  2.34s/it]

✓ Both pipelines loaded


## Part 2: Multi-Label Batch Classification

In [3]:
def classify_topics(pipeline, texts, labels, threshold=0.4):
    """Classify texts into topic labels. Returns list of label name lists."""
    all_results = pipeline(texts, labels, threshold=threshold, batch_size=4)
    return [[r['label'] for r in res] for res in all_results]


TOPIC_LABELS = ["technology", "finance", "sports", "science", "politics",
                "entertainment", "world news", "business", "health", "space"]

texts = [a["text"] for a in articles]
predictions = classify_topics(pipeline_ml, texts, TOPIC_LABELS)

assert isinstance(predictions, list) and len(predictions) == len(articles)
for pred in predictions:
    assert isinstance(pred, list)
assert "technology" in predictions[0]
assert "sports" in predictions[2]

print("✓ classify_topics works")
for article, pred in zip(articles, predictions):
    print(f"  [{article['domain']:12}] {pred}")

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:00<00:00,  2.55it/s]

100%|██████████| 3/3 [00:00<00:00,  7.16it/s]

✓ classify_topics works
  [tech        ] ['technology']
  [finance     ] ['finance', 'business']
  [sports      ] ['sports', 'space']
  [science     ] ['technology', 'science', 'health']
  [politics    ] ['finance', 'politics']
  [entertainment] ['technology', 'finance', 'entertainment', 'business']
  [world       ] ['health']
  [tech        ] ['technology']
  [space       ] ['technology', 'science', 'space']
  [sports      ] ['finance', 'sports', 'entertainment']


## Part 3: Single-Label Accuracy

In [4]:
TOPIC_LABELS_SL = ["technology", "finance", "sports", "science", "politics",
                   "entertainment", "world news", "business", "health", "space",
                   "disaster"]

results = pipeline_sl(texts, TOPIC_LABELS_SL, batch_size=4)
correct = sum(
    1 for article, res in zip(articles, results)
    if res[0]['label'] in article['expected_topics']
)
accuracy = correct / len(articles)

assert 0.0 <= accuracy <= 1.0
assert accuracy >= 0.5, f"Accuracy {accuracy:.0%} too low"
print(f"✓ Single-label accuracy: {accuracy:.0%} ({correct}/{len(articles)})")

for article, res in zip(articles, results):
    predicted = res[0]['label']
    match = "✓" if predicted in article['expected_topics'] else "✗"
    print(f"  {match} [{article['domain']:12}] {predicted:15} expected={article['expected_topics']}")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 47.24it/s]

✓ Single-label accuracy: 100% (10/10)
  ✓ [tech        ] technology      expected=['technology']
  ✓ [finance     ] finance         expected=['finance', 'economics']
  ✓ [sports      ] sports          expected=['sports']
  ✓ [science     ] health          expected=['science', 'health']
  ✓ [politics    ] politics        expected=['politics']
  ✓ [entertainment] entertainment   expected=['entertainment', 'business']
  ✓ [world       ] disaster        expected=['disaster', 'world news']
  ✓ [tech        ] technology      expected=['technology', 'business']
  ✓ [space       ] space           expected=['science', 'space']
  ✓ [sports      ] sports          expected=['sports']
